In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils

In [ ]:
parallelism.set_max_num_tbb_threads(4)

In [ ]:
m, fuseMarkers, fuseEdges = wall_generation.triangulate_channel_walls(*parametric_pillows.concentricCircles(8, 100), 0.001)
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=5, height=5)

In [ ]:
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
import time, vis
benchmark.reset()
isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
isheet.pressure = 20 * 3.75
opts.niter = 2000
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update()
cr = inflation.inflation_newton(isheet, isheet.rigidMotionPinVars[:3], opts, callback=cb)
benchmark.report()

In [ ]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])

In [ ]:
import compute_vibrational_modes

In [ ]:
class ModalAnalysisWrapper:
    def __init__(self, sheet):
        self.sheet = sheet
    def hessian(self):
        return self.sheet.hessian(inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
# lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(ModalAnalysisWrapper(isheet), mtype=compute_vibrational_modes.MassMatrixType.FULL, n=16, sigma=-1e-10)

# import mode_viewer, importlib
# importlib.reload(mode_viewer);
# mview = mode_viewer.ModeViewer(isheet, modes, lambdas, amplitude=10)
# # mview.showScalarField(rod_colors)
# mview.show()

In [ ]:
Pressure = inflation.InflatableSheet.EnergyType.Pressure
Elastic  = inflation.InflatableSheet.EnergyType.Elastic
Full     = inflation.InflatableSheet.EnergyType.Full

In [ ]:
fd_validation.gradConvergencePlot(isheet, customArgs = {"energyType": Full})

In [ ]:
fd_validation.gradConvergencePlot(isheet, customArgs = {"energyType": Elastic})

In [ ]:
fd_validation.gradConvergencePlot(isheet, customArgs = {"energyType": Pressure})

In [ ]:
fd_validation.hessConvergencePlot(isheet, customArgs = {"energyType": Full})

In [ ]:
fd_validation.hessConvergencePlot(isheet, customArgs = {"energyType": Elastic})

In [ ]:
fd_validation.hessConvergencePlot(isheet, customArgs = {"energyType": Pressure})